In [ ]:
import logging
import os
import re
import time

import nest_asyncio

from gmas.builder import GraphBuilder
from gmas.execution import MACPRunner, RunnerConfig, StreamEventType
from gmas.tools import WebSearchTool, create_openai_caller, register_tool

os.environ.setdefault("NO_PROXY", "*")  # bypass Windows system proxy
logging.disable(logging.INFO)  # silence noisy HTTP / framework logs
nest_asyncio.apply()

# ── Shared LLM caller ────────────────────────────────────────────
caller = create_openai_caller(
    base_url="",
    api_key="",
    model="",
    temperature=0,
    max_tokens=4_000,
    tool_choice="auto",
)


# ── Helper: run query and print answer + visited URLs ─────────────────
def _run(graph, runner, query):
    graph.update_task(query)
    urls = []
    t0 = time.perf_counter()
    for ev in runner.stream(graph):
        if ev.event_type == StreamEventType.AGENT_OUTPUT and ev.content:
            urls += re.findall(r'https?://[^\s\]"<>)]+', ev.content)
        elif ev.event_type == StreamEventType.RUN_END:
            wall = time.perf_counter() - t0
            print(ev.final_answer)
            urls += re.findall(r'https?://[^\s\]"<>)]+', ev.final_answer or "")
            seen = dict.fromkeys(urls)
            if seen:
                print(f"\n🌐 Sources ({len(seen)}):")
                for i, u in enumerate(seen, 1):
                    print(f"   {i}. {u}")
            print(f"\n⏱ {wall:.1f}s wall | {ev.total_time:.1f}s runner | tokens: {ev.total_tokens}")

In [ ]:
# ── Web Search agent ──────────────────────────────────────────────────────────
register_tool(
    WebSearchTool(
        max_results=10,
        max_content_length=12000,
        fetch_content=True,
        max_fetch_pages=10,
        timeout=15,
    )
)

PERSONA = """
You are a fast web research agent. Answer the user's question concisely.
"""
DESCRIPTION = """
perform an analysis to answer the user's question using web search for information,
do not invent anything, use only information from the Internet, and search until you find what you need.
REQUIRED: additionally, write down the relevant links that you visited for the search.
"""
b = GraphBuilder()
b.add_agent("researcher", persona=PERSONA, description=DESCRIPTION, tools=["web_search"])
b.add_task(query="(placeholder)")
b.connect_task_to_agents(agent_ids=["researcher"])
graph = b.build()
runner = MACPRunner(llm_caller=caller, config=RunnerConfig(max_tool_iterations=2, timeout=60))


def run_query(q):
    _run(graph, runner, q)


print("✅ web_search ready")

In [ ]:
run_query(
    "What are the latest papers on agent systems from the past week as of March 16, 2026? "
    "Only include current results from that week, and prioritize arXiv."
)

In [ ]:
# ── Deep Search agent (web_search + Selenium) ────────────────────────────────
from gmas.tools import ToolRegistry

deep_registry = ToolRegistry()
_deep_tool = WebSearchTool(
    max_results=10,
    max_content_length=12000,
    fetch_content=True,
    max_fetch_pages=5,
    timeout=25,
    deep_search="selenium",
    browser_config={
        "headless": True,
        "browser": "auto",
        "scroll_to_bottom": True,
        "max_scrolls": 3,
        "disable_images": True,
    },
)
_deep_tool.warm_up()
deep_registry.register(_deep_tool)

DEEP_PERSONA = """
You are a fast web research agent. Answer the user's question concisely.
"""
DEEP_DESCRIPTION = """
Analyze the user's question using web search. Do not invent information.
Include current links to every source you visited.
"""

b2 = GraphBuilder()
b2.add_agent("deep_researcher", persona=DEEP_PERSONA, description=DEEP_DESCRIPTION, tools=["web_search"])
b2.add_task(query="(placeholder)")
b2.connect_task_to_agents(agent_ids=["deep_researcher"])
deep_graph = b2.build()
deep_runner = MACPRunner(
    llm_caller=caller,
    config=RunnerConfig(max_tool_iterations=4, timeout=180, tool_registry=deep_registry),
)


def run_deep_query(q):
    _run(deep_graph, deep_runner, q)


print("✅ deep_search (Selenium) ready")

In [ ]:
run_deep_query(
    "What are the latest papers on agent systems from the past week as of March 16, 2026? "
    "Only include current results from that week, and prioritize arXiv."
)

In [ ]:
# ── Deep Search agent (web_search + Playwright) ──────────────────────────────
from gmas.tools import ToolRegistry

pw_registry = ToolRegistry()
_pw_tool = WebSearchTool(
    max_results=10,
    max_content_length=12000,
    fetch_content=True,
    max_fetch_pages=5,
    timeout=25,
    deep_search="playwright",
    browser_config={
        "headless": True,
        "browser": "chromium",
        "scroll_to_bottom": True,
        "max_scrolls": 3,
        "disable_images": True,
    },
)
_pw_tool.warm_up()
pw_registry.register(_pw_tool)

PW_PERSONA = """
You are a fast web research agent. Answer the user's question concisely.
"""
PW_DESCRIPTION = """
Analyze the user's question using web search. Do not invent information.
Include current links to every source you visited. If you visited no sources, do not list any.
"""

b3 = GraphBuilder()
b3.add_agent("pw_researcher", persona=PW_PERSONA, description=PW_DESCRIPTION, tools=["web_search"])
b3.add_task(query="(placeholder)")
b3.connect_task_to_agents(agent_ids=["pw_researcher"])
pw_graph = b3.build()
pw_runner = MACPRunner(
    llm_caller=caller,
    config=RunnerConfig(max_tool_iterations=4, timeout=180, tool_registry=pw_registry),
)


def run_pw_query(q):
    _run(pw_graph, pw_runner, q)


print("✅ deep_search (Playwright) ready")

In [ ]:
run_pw_query(
    "What are the latest papers on agent systems from the past week as of March 16, 2026? "
    "Only include current results from that week, and prioritize arXiv."
)

In [ ]:
run_pw_query("What is today's date and current time? Also check my internet connection speed.")